In [49]:
from git import Repo
import requests
import os

## Part 1: Individual fibonacci codes

In [38]:
with open('module1_basics/fibonacci_4.java', 'r') as file:
    contents = file.read()
    line_count = contents.count('\n')
    print("Total lines:", line_count)

Total lines: 82


In [39]:
contents = contents.split('\n')

In [40]:
logical_lines = line_count
contents_modified = []
for i in range(len(contents)):
    contents[i]=contents[i].replace(" ","")
    if len(contents[i])==0:
        #print(f"'{contents[i]}'  -- len=0")
        logical_lines-=1
    if len(contents[i])>0 and contents[i][0]=='/':
        #print(f"'{contents[i]}'  -- comment")
        logical_lines-=1
    #l= any(c.isalpha() for c in contents[i])
    #n= any(c.isdigit() for c in contents[i])
    
    #if (n or l):
    #    nb_lines+=1
logical_lines


49

In [41]:
with open('module1_basics/fibonacci_4.java', 'r') as file:
    contents = file.read()

In [42]:
contents = contents.split('\n')

In [43]:
mccabe = 1
for i in range(len(contents)):
    if 'if ' in contents[i] or 'for ' in contents[i] or 'while ' in contents[i] or 'case ' in contents[i] or '&&' in contents[i] or '||' in contents[i]:
        mccabe+=1
mccabe

11

## Part 2: Code base tests

In [48]:
def get_git_branches(repo_url):
    api_url = f"{repo_url.rstrip('/')}/info/refs?service=git-upload-pack"
    response = requests.get(api_url, stream=True)
    print(api_url)
    print(response)

    branches = []
    for line in response.iter_lines(decode_unicode=True):
        if line.startswith("refs/heads/"):
            branch_name = line.split("refs/heads/")[-1]
            branches.append(branch_name)
    
    return branches

In [ ]:
clone_dir_path = "/Users/lamartp/Documents/linux"

In [46]:
Repo.clone_from("https://github.com/torvalds/linux", clone_dir_path)

<git.repo.base.Repo 'c:\\Users\\lamartp\\Documents\\linux\\.git'>

In [47]:
repo = Repo(clone_dir_path)

In [50]:
#create a list of all files in the repository
file_paths = []
for root, dirs, files in os.walk(repo.working_tree_dir):
    for file in files:
        if file.endswith('.c') or file.endswith('.java') or file.endswith('.py'):
            file_paths.append(os.path.relpath(os.path.join(root, file), repo.working_tree_dir))
            #print(os.path.relpath(os.path.join(root, file), repo.working_tree_dir))

In [51]:
# create a dictionary to store metrics for each file
file_metrics = {}

# initialize the global metric evaluations
total_phys_lines = 0
total_logical_lines = 0
total_mccabe = 0
total_fan_in = 0

# get the names of files in the repository
file_names = [os.path.basename(path) for path in file_paths]

# create an array to store fan-out counts for each file
fan_out = [0]*len(file_names)

for file in file_paths:
    # get the file path
    file_to_read = os.path.join(clone_dir_path, file)
    with open(file_to_read) as f:
        # read the content
        contents = f.read()

        # initialize fan in for each file
        fan_in = 0

        # count the physical lines
        phys_lines = contents.count('\n')

        # initialize logical lines, the value will be decremented
        logical_lines = phys_lines

        # for every file in the repository, check if its name appears in the content of the current file
        for j in range(len(file_names)):
            # if the name appears, increment the fan-out for that file and the fan-in for the current file
            if file_names[j] in contents:
                fan_out[j]+=1
                fan_in+=1

        # split the content to reade the code line by line
        contents = contents.split('\n')

        # initialize mccabe complexity for each file
        mccabe = 1

        for i in range(len(contents)):
            # if we have a branching statement, increment mccabe complexity
            if 'if ' in contents[i] or 'for ' in contents[i] or 'while ' in contents[i] or 'case ' in contents[i] or '&&' in contents[i] or '||' in contents[i]:
                mccabe+=1

            # remove spaces to avoid indents when checking for non-logical lines
            contents[i]=contents[i].replace(" ","")
            # if the line is empty, decrement logical lines
            if len(contents[i])==0:
                logical_lines-=1
            # if the line is a comment, decrement logical lines
            if file.endswith('.c') and len(contents[i])>0 and contents[i][0]=='/':
                logical_lines-=1
            if file.endswith('.java') and len(contents[i])>0 and contents[i][0]=='/':
                logical_lines-=1
            if file.endswith('.py') and len(contents[i])>0 and contents[i][0]=='#':
                logical_lines-=1
        
        # create a temporary dictionary to store metrics for the current file
        dict = {}
        dict["physical_lines"] = phys_lines
        dict["logical_lines"] = logical_lines
        dict["mccabe"] = mccabe
        dict["fan_in"] = fan_in
        # store the metrics in the file_metrics dictionary
        file_metrics[file] = dict

        # update the global metrics
        total_phys_lines += phys_lines
        total_logical_lines += logical_lines
        total_mccabe += mccabe
        total_fan_in += fan_in

In [52]:
total_phys_lines, total_logical_lines, total_mccabe, total_fan_in

(26843554, 22608829, 2541899, 325930)

In [53]:
for file in file_paths:
    file_metrics[file]["fan_out"] = fan_out[file_paths.index(file)]

In [54]:
total_fan_out = sum(fan_out)
total_fan_out

325930

In [55]:
file_metrics

{'arch\\alpha\\boot\\bootp.c': {'physical_lines': 214,
  'logical_lines': 177,
  'mccabe': 10,
  'fan_in': 119,
  'fan_out': 1},
 'arch\\alpha\\boot\\bootpz.c': {'physical_lines': 475,
  'logical_lines': 380,
  'mccabe': 34,
  'fan_in': 116,
  'fan_out': 1},
 'arch\\alpha\\boot\\main.c': {'physical_lines': 190,
  'logical_lines': 158,
  'mccabe': 14,
  'fan_in': 115,
  'fan_out': 102},
 'arch\\alpha\\boot\\misc.c': {'physical_lines': 172,
  'logical_lines': 132,
  'mccabe': 12,
  'fan_in': 36,
  'fan_out': 33},
 'arch\\alpha\\boot\\stdio.c': {'physical_lines': 302,
  'logical_lines': 266,
  'mccabe': 79,
  'fan_in': 0,
  'fan_out': 0},
 'arch\\alpha\\boot\\tools\\mkbb.c': {'physical_lines': 153,
  'logical_lines': 118,
  'mccabe': 2,
  'fan_in': 0,
  'fan_out': 0},
 'arch\\alpha\\boot\\tools\\objstrip.c': {'physical_lines': 284,
  'logical_lines': 240,
  'mccabe': 42,
  'fan_in': 1,
  'fan_out': 1},
 'arch\\alpha\\kernel\\asm-offsets.c': {'physical_lines': 33,
  'logical_lines': 25,
  